In [1]:
from pathlib import Path
from datetime import datetime
import json
import re
import copy
import pandas as pd

# Resolve project root whether Jupyter starts in the project root or notebooks/.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CV_OUTPUT = DATA_PROCESSED / "cv_builder"
RESUME_LIBRARY = CV_OUTPUT / "resume_library"
TAILORED_FOLDER = CV_OUTPUT / "tailored_resumes"

for folder in (CV_OUTPUT, RESUME_LIBRARY, TAILORED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)

MODEL_PATH = PROJECT_ROOT / "models"
SELECTED_RESUME_ID = None  # Set to a saved candidate_id to select a specific original profile.

print("Project root:", PROJECT_ROOT)
print("Original Resume Library:", RESUME_LIBRARY)
print("Tailored variants:", TAILORED_FOLDER)

Project root: /Users/rizwansalim/Desktop/SkillBridgeByZequa
Original Resume Library: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/cv_builder/resume_library
Tailored variants: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/cv_builder/tailored_resumes


In [2]:
# Standard CV schema used by this notebook and downstream SkillBridge components.
resume_template = {
    "personal_details": {
        "full_name": "",
        "email": "",
        "phone": "",
        "location": "",
        "linkedin": "",
        "portfolio": "",
        "professional_summary": ""
    },
    "education": [],
    "experience": [],
    "projects": [],
    "skills": [],
    "certifications": [],
    "languages": [],
    "candidate_id": ""
}

with (CV_OUTPUT / "resume_template.json").open("w", encoding="utf-8") as f:
    json.dump(resume_template, f, indent=2, ensure_ascii=False)

cv_templates = {
    "ats_friendly": {
        "name": "ATS-Friendly", "layout": "single_column", "font": "Helvetica",
        "color": "#000000",
        "sections": ["personal_details", "professional_summary", "skills", "experience", "projects", "education", "certifications", "languages"]
    },
    "modern_professional": {
        "name": "Modern Professional", "layout": "professional", "font": "Helvetica",
        "color": "#1F3A5F",
        "sections": ["personal_details", "professional_summary", "experience", "skills", "education", "projects", "certifications", "languages"]
    },
    "technical_resume": {
        "name": "Technical Resume", "layout": "technical", "font": "Helvetica",
        "color": "#17365D",
        "sections": ["personal_details", "professional_summary", "technical_skills", "projects", "experience", "education", "certifications", "languages"]
    }
}

with (CV_OUTPUT / "cv_templates.json").open("w", encoding="utf-8") as f:
    json.dump(cv_templates, f, indent=2, ensure_ascii=False)

print("CV templates ready:", ", ".join(cv_templates))

CV templates ready: ats_friendly, modern_professional, technical_resume


In [3]:
def _as_text(value):
    """Normalize a scalar or list-like contact/summary value to a string."""
    if value is None:
        return ""
    if isinstance(value, (list, tuple)):
        return ", ".join(str(x).strip() for x in value if x is not None and str(x).strip())
    return str(value).strip()


def _as_items(value):
    """Normalize resume sections to a list without discarding source text."""
    if value is None or value == "":
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, str):
        text = value.strip()
        return [text] if text else []
    return [value]


def import_existing_resume(json_path):
    """Import either Notebook 08's candidate_profile schema or legacy flat JSON."""
    json_path = Path(json_path)
    if not json_path.is_file():
        raise FileNotFoundError(f"Resume JSON not found: {json_path}")

    with json_path.open("r", encoding="utf-8") as f:
        source = json.load(f)
    if not isinstance(source, dict):
        raise ValueError(f"Resume JSON must contain an object: {json_path}")

    # New Notebook 08 format: candidate_id/name/contact-like top-level fields.
    if any(k in source for k in ("candidate_id", "name", "summary", "source_file")):
        details = {
            "full_name": _as_text(source.get("name")),
            "email": _as_text(source.get("email")),
            "phone": _as_text(source.get("phone")),
            "location": _as_text(source.get("location")),
            "linkedin": _as_text(source.get("linkedin")),
            "portfolio": _as_text(source.get("portfolio")),
            "professional_summary": _as_text(source.get("summary"))
        }
        imported = copy.deepcopy(resume_template)
        imported["personal_details"] = details
        imported["candidate_id"] = _as_text(source.get("candidate_id")) or json_path.stem
        for section in ("education", "experience", "projects", "certifications", "languages"):
            imported[section] = _as_items(source.get(section))
        imported["skills"] = [str(x).strip() for x in _as_items(source.get("skills")) if str(x).strip()]
        return imported

    # Legacy Notebook 09 / generic CV-builder format.
    imported = copy.deepcopy(resume_template)
    personal = source.get("personal_details", {}) if isinstance(source.get("personal_details"), dict) else {}
    imported["personal_details"] = {
        key: _as_text(personal.get(key, source.get(key, "")))
        for key in resume_template["personal_details"]
    }
    imported["candidate_id"] = _as_text(source.get("candidate_id")) or json_path.stem
    for section in ("education", "experience", "projects", "certifications", "languages"):
        imported[section] = _as_items(source.get(section))
    imported["skills"] = [str(x).strip() for x in _as_items(source.get("skills")) if str(x).strip()]
    return imported


# Always define the list, even when the directory is empty (prevents NameError).
resume_json_files = sorted(
    [p for p in RESUME_LIBRARY.glob("*.json") if p.name != "resume_library_index.json"],
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if resume_json_files:
    if SELECTED_RESUME_ID:
        matches = [p for p in resume_json_files if p.stem == SELECTED_RESUME_ID]
        if not matches:
            raise FileNotFoundError(f"SELECTED_RESUME_ID={SELECTED_RESUME_ID!r} was not found in {RESUME_LIBRARY}")
        selected_resume_path = matches[0]
    else:
        selected_resume_path = resume_json_files[0]
    original_candidate = import_existing_resume(selected_resume_path)
    print("Loaded original profile:", selected_resume_path.name)
else:
    selected_resume_path = None
    original_candidate = copy.deepcopy(resume_template)
    original_candidate["personal_details"]["full_name"] = "Rizwan Salim"
    original_candidate["candidate_id"] = "rizwan-salim-starter"
    print("No saved resume profile found. Using an empty Rizwan Salim starter profile for preview/testing.")
    print("Run Notebook 08 first to import a real resume profile.")

# Keep a separate working copy; never edit the original profile JSON in-place.
updated_candidate = copy.deepcopy(original_candidate)
print("Candidate:", updated_candidate["personal_details"].get("full_name") or "Name not supplied")
print("Skills loaded:", len(updated_candidate["skills"]))

Loaded original profile: candidate-sample-resume-2.json
Candidate: Name not supplied
Skills loaded: 5


In [4]:
def update_personal_details(resume, updates):
    """Update allowed fields on a working copy of a CV profile."""
    result = copy.deepcopy(resume)
    allowed = set(resume_template["personal_details"])
    for field, value in updates.items():
        if field in allowed:
            result["personal_details"][field] = _as_text(value)
    return result


def update_resume_sections(resume, experience=None, projects=None, skills=None,
                           education=None, certifications=None, languages=None):
    result = copy.deepcopy(resume)
    for key, value in {
        "experience": experience, "projects": projects, "skills": skills,
        "education": education, "certifications": certifications, "languages": languages
    }.items():
        if value is not None:
            result[key] = _as_items(value)
    return result


def generate_cv(candidate_data, template_name="ats_friendly"):
    if template_name not in cv_templates:
        raise ValueError(f"Unknown CV template: {template_name}. Choose from {list(cv_templates)}")
    template = cv_templates[template_name]
    sections = {}
    for section in template["sections"]:
        if section == "technical_skills":
            sections[section] = list(candidate_data.get("skills", []))
        elif section == "professional_summary":
            sections[section] = candidate_data.get("personal_details", {}).get("professional_summary", "")
        elif section in candidate_data:
            sections[section] = candidate_data[section]
        elif section == "personal_details":
            sections[section] = candidate_data.get("personal_details", {})
    return {"candidate_id": candidate_data.get("candidate_id", ""),
            "template": template["name"], "layout": template["layout"],
            "color": template["color"], "sections": sections}


generated_cv = generate_cv(updated_candidate, template_name="ats_friendly")
generated_cv_path = CV_OUTPUT / "generated_cv.json"
with generated_cv_path.open("w", encoding="utf-8") as f:
    json.dump(generated_cv, f, indent=2, ensure_ascii=False)
print("CV JSON generated:", generated_cv_path)

CV JSON generated: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/cv_builder/generated_cv.json


In [5]:
from collections import Counter


def extract_ats_keywords(text):
    words = re.findall(r"(?u)\b[\w][\w+#.-]*\b", str(text).lower())
    stopwords = {
        "the", "and", "for", "with", "you", "are", "our", "this", "that",
        "will", "have", "from", "your", "work", "ability", "using", "who",
        "to", "of", "in", "a", "an", "on", "as", "is", "be", "or", "by"
    }
    return Counter(w for w in words if w not in stopwords and len(w) > 1)


def _item_to_text(item):
    if isinstance(item, dict):
        return " ".join(str(v) for v in item.values() if v is not None)
    return str(item)


def create_resume_text(resume):
    details = resume.get("personal_details", {})
    parts = [details.get("professional_summary", "")]
    for key in ("skills", "experience", "projects", "education", "certifications", "languages"):
        parts.extend(_item_to_text(item) for item in resume.get(key, []))
    return " ".join(str(part) for part in parts if part)


# Example job description for demonstration only; replace with a real target description.
job_description = """
Data Scientist role: Python, SQL, machine learning, deep learning, NLP,
Pandas, NumPy, scikit-learn, data visualization, predictive modelling and data analysis.
"""

resume_keywords = extract_ats_keywords(create_resume_text(updated_candidate))
job_keywords = extract_ats_keywords(job_description)
matched_keywords = sorted(set(resume_keywords) & set(job_keywords))
missing_keywords = sorted(set(job_keywords) - set(resume_keywords))
ats_keyword_coverage = (len(matched_keywords) / len(job_keywords) * 100) if job_keywords else 0.0

print("Matched job-description keywords:", matched_keywords)
print("Missing job-description keywords:", missing_keywords)
print(f"ATS keyword coverage: {ats_keyword_coverage:.2f}%")

Matched job-description keywords: ['deep', 'learning', 'machine', 'sql']
Missing job-description keywords: ['analysis', 'data', 'modelling', 'nlp', 'numpy', 'pandas', 'predictive', 'python', 'role', 'scientist', 'scikit-learn', 'visualization']
ATS keyword coverage: 25.00%


In [6]:
def calculate_resume_score(resume, ats_coverage=0):
    details = resume.get("personal_details", {})
    scores = {}
    fields = ["full_name", "email", "phone", "location"]
    scores["Personal Details"] = sum(bool(details.get(f)) for f in fields) / len(fields) * 20
    summary = details.get("professional_summary", "")
    scores["Professional Summary"] = 15 if len(summary.split()) >= 20 else (7 if summary else 0)
    scores["Education"] = 15 if resume.get("education") else 0
    scores["Experience"] = 15 if resume.get("experience") else 0
    scores["Projects"] = 15 if resume.get("projects") else 0
    scores["Skills"] = min(len(resume.get("skills", [])) * 2, 10)
    scores["ATS Keywords"] = max(0.0, min(float(ats_coverage), 100.0)) / 100 * 10
    return {"section_scores": scores, "total_score": round(sum(scores.values()), 2), "maximum_score": 100}


resume_score = calculate_resume_score(updated_candidate, ats_coverage=ats_keyword_coverage)
score_path = CV_OUTPUT / "resume_score_report.json"
with score_path.open("w", encoding="utf-8") as f:
    json.dump(resume_score, f, indent=2, ensure_ascii=False)
print(json.dumps(resume_score, indent=2))

{
  "section_scores": {
    "Personal Details": 0.0,
    "Professional Summary": 0,
    "Education": 0,
    "Experience": 0,
    "Projects": 0,
    "Skills": 10,
    "ATS Keywords": 2.5
  },
  "total_score": 12.5,
  "maximum_score": 100
}


In [7]:
# Export dependencies are checked explicitly; install in the active Jupyter kernel if needed:
# %pip install reportlab python-docx
try:
    from reportlab.lib.pagesizes import A4
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib import colors
    from reportlab.lib.units import inch
    from xml.sax.saxutils import escape as xml_escape
except ImportError as exc:
    raise ImportError("PDF export requires ReportLab. Run: %pip install reportlab") from exc

try:
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
except ImportError as exc:
    raise ImportError("DOCX export requires python-docx. Run: %pip install python-docx") from exc


def export_cv_pdf(resume, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle("SkillBridgeCVTitle", parent=styles["Title"], fontSize=20,
                                 textColor=colors.HexColor("#1F3A5F"), spaceAfter=8)
    heading_style = ParagraphStyle("SkillBridgeSection", parent=styles["Heading2"], fontSize=12,
                                   textColor=colors.HexColor("#1F3A5F"), spaceBefore=12, spaceAfter=5)
    body_style = styles["BodyText"]
    doc = SimpleDocTemplate(str(output_path), pagesize=A4, rightMargin=.65*inch,
                            leftMargin=.65*inch, topMargin=.55*inch, bottomMargin=.55*inch)
    story = []
    details = resume.get("personal_details", {})
    story.append(Paragraph(xml_escape(details.get("full_name") or "Candidate Name"), title_style))
    contact = [details.get(k, "") for k in ("email", "phone", "location", "linkedin", "portfolio")]
    contact = [xml_escape(str(x)) for x in contact if x]
    if contact:
        story.append(Paragraph(" | ".join(contact), body_style))
    story.append(HRFlowable(width="100%", color=colors.grey))
    summary = details.get("professional_summary", "")
    if summary:
        story.append(Paragraph("PROFESSIONAL SUMMARY", heading_style))
        story.append(Paragraph(xml_escape(str(summary)).replace("\n", "<br/>"), body_style))
    for heading, key in [("SKILLS", "skills"), ("EXPERIENCE", "experience"), ("PROJECTS", "projects"),
                         ("EDUCATION", "education"), ("CERTIFICATIONS", "certifications"), ("LANGUAGES", "languages")]:
        items = resume.get(key, [])
        if not items:
            continue
        story.append(Paragraph(heading, heading_style))
        for item in items:
            if isinstance(item, dict):
                for field, value in item.items():
                    if value:
                        line = f"<b>{xml_escape(str(field).replace('_', ' ').title())}:</b> {xml_escape(str(value))}"
                        story.append(Paragraph(line, body_style))
            else:
                for line in str(item).splitlines() or [str(item)]:
                    if line.strip():
                        story.append(Paragraph(xml_escape(line.strip()), body_style))
            story.append(Spacer(1, 5))
    doc.build(story)
    return output_path


def export_cv_docx(resume, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    doc = Document()
    section = doc.sections[0]
    section.top_margin = Inches(.6); section.bottom_margin = Inches(.6)
    section.left_margin = Inches(.7); section.right_margin = Inches(.7)
    details = resume.get("personal_details", {})
    title = doc.add_heading(details.get("full_name") or "Candidate Name", level=0)
    for run in title.runs:
        run.font.color.rgb = RGBColor(31, 58, 95); run.font.size = Pt(22)
    contact = [details.get(k, "") for k in ("email", "phone", "location", "linkedin", "portfolio")]
    contact = [str(x) for x in contact if x]
    if contact: doc.add_paragraph(" | ".join(contact))
    summary = details.get("professional_summary", "")
    if summary:
        doc.add_heading("Professional Summary", level=1); doc.add_paragraph(str(summary))
    for heading, key in [("Skills", "skills"), ("Experience", "experience"), ("Projects", "projects"),
                         ("Education", "education"), ("Certifications", "certifications"), ("Languages", "languages")]:
        items = resume.get(key, [])
        if not items: continue
        doc.add_heading(heading, level=1)
        for item in items:
            if isinstance(item, dict):
                for field, value in item.items():
                    if value:
                        p = doc.add_paragraph(); r = p.add_run(f"{str(field).replace('_', ' ').title()}: "); r.bold = True
                        p.add_run(str(value))
            else:
                for line in str(item).splitlines() or [str(item)]:
                    if line.strip(): doc.add_paragraph(line.strip(), style="List Bullet")
    doc.save(str(output_path))
    return output_path


def preview_resume(resume):
    from IPython.display import display, HTML
    from html import escape as html_escape
    details = resume.get("personal_details", {})
    name = html_escape(details.get("full_name") or "Candidate Name")
    contact = " | ".join(html_escape(str(details.get(k, ""))) for k in ("email", "phone", "location") if details.get(k))
    html = [f"<div style='max-width:760px;margin:auto;padding:28px;border:1px solid #ddd;font-family:Arial;color:#333'><h1 style='color:#1F3A5F'>{name}</h1><p>{contact}</p>"]
    summary = details.get("professional_summary", "")
    if summary: html.append(f"<h3>Professional Summary</h3><p>{html_escape(str(summary))}</p>")
    for heading, key in [("Skills", "skills"), ("Experience", "experience"), ("Projects", "projects"), ("Education", "education"), ("Certifications", "certifications"), ("Languages", "languages")]:
        items = resume.get(key, [])
        if not items: continue
        html.append(f"<h3 style='color:#1F3A5F'>{heading}</h3>")
        for item in items:
            if isinstance(item, dict):
                html.append("<div>")
                for field, value in item.items():
                    if value: html.append(f"<p><b>{html_escape(str(field).replace('_',' ').title())}:</b> {html_escape(str(value))}</p>")
                html.append("</div>")
            else: html.append(f"<p>{html_escape(str(item)).replace(chr(10), '<br/>')}</p>")
    html.append("</div>")
    display(HTML("".join(html)))


pdf_path = CV_OUTPUT / "skillbridge_generated_cv.pdf"
docx_path = CV_OUTPUT / "skillbridge_generated_cv.docx"
export_cv_pdf(updated_candidate, pdf_path)
export_cv_docx(updated_candidate, docx_path)
print("PDF exported:", pdf_path, pdf_path.exists())
print("DOCX exported:", docx_path, docx_path.exists())

PDF exported: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/cv_builder/skillbridge_generated_cv.pdf True
DOCX exported: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/cv_builder/skillbridge_generated_cv.docx True


## Role-specific resume variants

Select an original profile by its `candidate_id`, then call `generate_role_tailored_resume(...)`. The function creates a separate JSON/PDF/DOCX variant and does not overwrite the original profile. It ranks only skills already present in the original profile using keyword matches from the job description.

In [8]:
def _role_slug(value):
    slug = re.sub(r"[^a-z0-9]+", "-", str(value).lower()).strip("-")
    return slug or "target-role"


def _load_original_profile(resume_id):
    candidates = [p for p in RESUME_LIBRARY.glob("*.json") if p.name != "resume_library_index.json"]
    matches = [p for p in candidates if p.stem == resume_id]
    if not matches:
        raise FileNotFoundError(f"Original resume ID {resume_id!r} was not found in {RESUME_LIBRARY}")
    return import_existing_resume(matches[0]), matches[0]


def generate_role_tailored_resume(resume_id, target_role, job_description, template_name="ats_friendly"):
    """Save a separate role-tailored variant; never changes the original resume."""
    if not str(target_role).strip():
        raise ValueError("target_role cannot be empty")
    if not str(job_description).strip():
        raise ValueError("job_description cannot be empty")

    original, original_path = _load_original_profile(resume_id)
    tailored = copy.deepcopy(original)
    jd_text = str(job_description).lower()
    original_skills = list(original.get("skills", []))
    # Stable ranking: skills explicitly mentioned in the JD first; all remaining original skills retained.
    ranked = sorted(enumerate(original_skills),
                    key=lambda pair: (0 if re.search(r"(?<![a-z0-9])" + re.escape(str(pair[1]).lower()) + r"(?![a-z0-9])", jd_text) else 1, pair[0]))
    tailored["skills"] = [skill for _, skill in ranked]
    tailored["target_role"] = str(target_role).strip()
    tailored["source_resume_id"] = str(resume_id)
    tailored["tailoring_note"] = "Existing skills reordered by job-description keyword matches; no new skills or experience added."

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    variant_id = f"{_role_slug(resume_id)}__{_role_slug(target_role)}__{stamp}"
    tailored["variant_id"] = variant_id
    TAILORED_FOLDER.mkdir(parents=True, exist_ok=True)
    json_out = TAILORED_FOLDER / f"{variant_id}.json"
    pdf_out = TAILORED_FOLDER / f"{variant_id}.pdf"
    docx_out = TAILORED_FOLDER / f"{variant_id}.docx"
    with json_out.open("w", encoding="utf-8") as f:
        json.dump(tailored, f, indent=2, ensure_ascii=False)
    export_cv_pdf(tailored, pdf_out)
    export_cv_docx(tailored, docx_out)

    index_path = CV_OUTPUT / "resume_library_index.json"
    index = {"original_profiles": [], "tailored_variants": []}
    if index_path.exists():
        try:
            with index_path.open("r", encoding="utf-8") as f: index = json.load(f)
        except (json.JSONDecodeError, OSError):
            pass
    if not isinstance(index, dict): index = {"original_profiles": [], "tailored_variants": []}
    index.setdefault("original_profiles", [])
    index.setdefault("tailored_variants", [])
    index["tailored_variants"].append({"variant_id": variant_id, "resume_id": str(resume_id),
                                       "target_role": str(target_role).strip(), "job_description": str(job_description),
                                       "json": str(json_out), "pdf": str(pdf_out), "docx": str(docx_out),
                                       "created_at": datetime.now().isoformat(timespec="seconds")})
    with index_path.open("w", encoding="utf-8") as f:
        json.dump(index, f, indent=2, ensure_ascii=False)
    files = {"variant_id": variant_id, "json": json_out, "pdf": pdf_out, "docx": docx_out,
             "source_original": original_path}
    return files, tailored


print("Role-tailoring function ready.")
print("Original profile count:", len(resume_json_files))

Role-tailoring function ready.
Original profile count: 4


In [9]:
# Save the current CV-builder working copy as a generated artifact (not as an original resume).
updated_candidate_path = CV_OUTPUT / "updated_candidate.json"
with updated_candidate_path.open("w", encoding="utf-8") as f:
    json.dump(updated_candidate, f, indent=2, ensure_ascii=False)

# Refresh index metadata for original profiles without changing their files.
index_path = CV_OUTPUT / "resume_library_index.json"
if index_path.exists():
    try:
        with index_path.open("r", encoding="utf-8") as f: library_index = json.load(f)
    except (json.JSONDecodeError, OSError):
        library_index = {}
else:
    library_index = {}
if not isinstance(library_index, dict): library_index = {}
library_index.setdefault("original_profiles", [])
library_index.setdefault("tailored_variants", [])
known_ids = {item.get("resume_id") for item in library_index["original_profiles"] if isinstance(item, dict)}
for path in resume_json_files:
    if path.stem not in known_ids:
        try:
            with path.open("r", encoding="utf-8") as f: raw = json.load(f)
            library_index["original_profiles"].append({"resume_id": raw.get("candidate_id", path.stem), "file": str(path)})
        except (json.JSONDecodeError, OSError):
            print("Skipped unreadable resume JSON:", path.name)
with index_path.open("w", encoding="utf-8") as f:
    json.dump(library_index, f, indent=2, ensure_ascii=False)

preview_resume(updated_candidate)

In [10]:
# Final integrity checks and artifact report.
required_files = {
    "Resume template": CV_OUTPUT / "resume_template.json",
    "CV templates": CV_OUTPUT / "cv_templates.json",
    "Generated CV JSON": generated_cv_path,
    "Updated candidate JSON": updated_candidate_path,
    "Resume score report": score_path,
    "Resume library index": index_path,
    "PDF CV": pdf_path,
    "DOCX CV": docx_path
}

assert isinstance(updated_candidate.get("personal_details"), dict)
assert isinstance(updated_candidate.get("skills"), list)
assert isinstance(updated_candidate.get("experience"), list)
assert isinstance(updated_candidate.get("projects"), list)
assert 0 <= ats_keyword_coverage <= 100
assert 0 <= resume_score["total_score"] <= 100
for label, path in required_files.items():
    assert path.is_file(), f"Missing generated artifact: {label} -> {path}"
    print(f"FOUND | {label} | {path.name}")

print("\nSKILLBRIDGE NOTEBOOK 09 CHECKS PASSED")
print("Candidate:", updated_candidate["personal_details"].get("full_name") or "Not supplied")
print("Original profiles discovered:", len(resume_json_files))
print("ATS keyword coverage:", round(ats_keyword_coverage, 2), "%")
print("Resume score:", resume_score["total_score"], "/100")
print("PDF and DOCX exports verified.")

FOUND | Resume template | resume_template.json
FOUND | CV templates | cv_templates.json
FOUND | Generated CV JSON | generated_cv.json
FOUND | Updated candidate JSON | updated_candidate.json
FOUND | Resume score report | resume_score_report.json
FOUND | Resume library index | resume_library_index.json
FOUND | PDF CV | skillbridge_generated_cv.pdf
FOUND | DOCX CV | skillbridge_generated_cv.docx

SKILLBRIDGE NOTEBOOK 09 CHECKS PASSED
Candidate: Not supplied
Original profiles discovered: 4
ATS keyword coverage: 25.0 %
Resume score: 12.5 /100
PDF and DOCX exports verified.
